# NHS England Maternal Care Pathways Master Pipeline
## Stage 8 - Define final pre-eclampsia diagnosis, stratify, and compute continuous IMD score

### Compiled by Federica Caretta Cortegiani

In [0]:
import sys
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql import SparkSession 
from pyspark.sql.functions import when, col, lit, count, max, last, first, min, avg, sum, collect_list, collect_set, countDistinct, to_date, datediff, array_contains, size, udf, format_number, regexp_replace, explode, array, array_union, desc_nulls_last, flatten, split, filter, floor, ceil
from pyspark.sql.types import IntegerType, StringType, NumericType, DateType, ArrayType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.ml.regression import GeneralizedLinearRegression as GLR

%run "/Workspace/Shared/Helper_Methods_EP"


In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
read_table_name_1 = "msds_diag_busy_filtered_final_imputed_reduced_stage"
read_table_name_2 = "msds_diag_busy_filtered_final_imputed_preeclampsia_diag_stage"

df_master = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{read_table_name_1}")
df_preeclampsia_diag = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{read_table_name_2}")

In [0]:
df_master = (
    df_master
    .join(
        df_preeclampsia_diag,
        on="UniqPregID",
        how="left"
    )
)

# Final Preeclampsia Diagnosis

In [0]:
df_master = (
    df_master
    .withColumns({
        "Possible_preeclampsia_diagnosis_OR": when(((col("Hypertension_2x") == 1) | (col("Proteinuria_1x") == 1) | (col("PlGF_test") == 1)), 1).otherwise(0),
        "Possible_preeclampsia_diagnosis_AND": when(((col("Hypertension_2x") == 1) & (col("Proteinuria_1x") == 1)), 1).otherwise(0),
        "Possible_preeclampsia_diagnosis_AND1x": when(((col("Hypertension_count") >= 1) & (col("Proteinuria_count") >= 1)), 1).otherwise(0),
        "Possible_preeclampsia_diagnosis_AND1x_or_Hyp2x": when((((col("Hypertension_count") >= 1) & (col("Proteinuria_count") >= 1)) | (col("Hypertension_count") >= 2)), 1).otherwise(0),
        "Possible_preeclampsia_diagnosis_AND2x": when(((col("Hypertension_count") >= 2) & (col("Proteinuria_count") >= 2)), 1).otherwise(0)
    })
)

In [0]:
# Preeclampsia_during_this_pregnancy & possible_preeclampsia_diagnosis_AND1x_or_Hyp2x
# Early: set to 0 if overall 0
# Late: set 0 if overall 0, set to null if overall null

df_master_final = (
    df_master
    .withColumn(
        "Preeclampsia_during_this_pregnancy",  when(((col("Preeclampsia_during_this_pregnancy") == 1) & (col("Possible_preeclampsia_diagnosis_AND1x_or_Hyp2x") == 1)), 1).when(col("Preeclampsia_during_this_pregnancy").isNull(), None).otherwise(0)
    )
    .withColumns({
        "Preeclampsia_early_onset": when(col("Preeclampsia_during_this_pregnancy") == 0, 0).otherwise(col("Preeclampsia_early_onset")),
        "Preeclampsia_late_onset": when(col("Preeclampsia_during_this_pregnancy") == 0, 0).when(col("Preeclampsia_during_this_pregnancy").isNull(), None).otherwise(col("Preeclampsia_late_onset")),
    })
)

In [0]:
print(f"Preeclampsia prevalence: {100 * df_master_final.filter(col("Preeclampsia_during_this_pregnancy") == 1).count() / df_master_final.filter(col("Preeclampsia_during_this_pregnancy").isNotNull()).count():.2f}%")
print(f"Early preeclampsia prevalence: {100 * df_master_final.filter(col("Preeclampsia_early_onset") == 1).count() / df_master_final.filter(col("Preeclampsia_early_onset").isNotNull()).count():.2f}%")
print(f"Late preeclampsia prevalence: {100 * df_master_final.filter(col("Preeclampsia_late_onset") == 1).count() / df_master_final.filter(col("Preeclampsia_late_onset").isNotNull()).count():.2f}%")

# Update smoking variables


In [0]:
SmokingFindingCode = "'697956009','56294008','30310000','110483000','ZV6D8','ZV4K0','ZRh4.','ZRh4.','ZRBm2','ZRBm2','ZRao.','ZRaM.','ZG233','Eu171','Eu17.','E251z','E2511','E2510','E251.','E023.','9OOZ.','9OOA.','9OO9.','9OO8.','9OO7.','9OO6.','9OO5.','9OO4.','9OO3.','9OO2.','9OO1.','9OO..','9N4M.','9N2k.','9ko..','9kc0.','9kc..','8I3M.','8I39.','8I2J.','8I2I.','8HTK.','8HkQ.','8HBM.','8H7i.','8CAL.','8CAg.','8BP3.','8B3Y.','8B3f.','8B2B.','745Hz','745Hy','745H4','745H3','745H2','745H1','745H0','745H.','67H6.','67A3.','38DH.','13p6.','13p5.','13p4.','13p3.','13p2.','13p1.','13p0.','13p..','137Z.','137Y.','137X.','137V.','137R.','137Q.','137Q.','137P.','137P.','137M.','137J.','137h.','137H.','137G.','137f.','137e.','137d.','137c.','137C.','137b.','137a.','1376.','1375.','1374.','1373.','1372.','Z720','Z716','F17','XE0or','XE0oq','XaWNE','XaLQh','XaJX2','XaItg','XaIIu','XagO3','Xaa26','Ub0pT','Ub0pJ','Eu17.','E251.','137R.','137M.','137G.','137C.','449345000','446172000','365982000','160616005','394871007','394873005','169940006','65568007','365981007','203191000000107','77176002','230065006','59978006','160605003','230063004','56771006','160603005','230060001','160604004','230062009','56578002','230059006','428041000124106','82302008','160619003','308438006','266929003','266920004','160606002','230064005'"

NonSmokingFindingCode = "'E2513','9km..','137T.','137S.','137O.','137N.','137K.','137j.','137F.','137B.','137A.','1379.','1378.','1377.','Xa1bv','Ub1na','137K.','9kn..','13WK.','137L.','8392000','XE0oh','137L.','1371.','908781000000104','722499006','266919005','449368009','8517006','405746006','360900008','360918006','160618006','160621008','281018007','266928006','266924008','1092071000000105','266922007','1092111000000104','266923002','1092091000000109','160620009','492191000000103','1092031000000108','735128000','48031000119106','266921000','1092131000000107','266925009','1092041000000104','360890004','360929005','105541001','105539002','105540000','53896009','87739003','786063001','785889008'"

NonSmokingObsCode = '160625004'
unknown_smoker_status_code = '266927001' # I'm going to ignore this one

cigsperday_obscode = ['230056004', '.137X', '137X.', 'Ub1tI']
carbon_monoxide_obscodes = ['251900003', '.4I90', '4I90.', 'X77Qd']

In [0]:
df_fo = spark.table(f'dsa_712819_x8g2j.dsa_712819_x8g2j_collab.msds_v2_findings_and_observations_all_years')
df_care = spark.table(f'dsa_712819_x8g2j.dsa_712819_x8g2j_collab.msds_v2_care_activities_all_years')

In [0]:
care_cols =[
    'UniqPregID', 'Person_ID_mother_DEID', 'obscode', 'obsvalue',
    'cigarettesperday', 'ccontactdate', 'comonreading', 'ucumunit',
    F.col("ccontactdate").alias('smoker_status_date') # rename this col
    ]

care_smoker_logic = (
    (F.col('cigarettesperday').try_cast('float') > 0) |
    (F.col('comonreading').try_cast('float') >= 4) |
    ((F.col('obscode').isin(cigsperday_obscode)) & (F.col('obsvalue').try_cast('float') > 0)) |
    ((F.col('obscode').isin(carbon_monoxide_obscodes)) & (F.upper(F.col('ucumunit')).like('%ppm%')) & (F.col('obsvalue').try_cast('float') >= 4))
)
care_non_smoker_logic = (
    (F.col('cigarettesperday').try_cast('float') == 0) |
    (F.col('obscode') == NonSmokingObsCode) |
    (F.col('comonreading').try_cast('float') < 4) |
    ((F.col('obscode').isin(cigsperday_obscode)) & (F.col('obsvalue').try_cast('float') == 0)) |
    ((F.col('obscode').isin(carbon_monoxide_obscodes)) & (F.upper(F.col('ucumunit')).like('%ppm%')) & (F.col('obsvalue').try_cast('float') < 4))
)

df_care_smoking_status_with_missing = df_care.select(
    *care_cols,
    F.when(care_smoker_logic, 'smoker')
    .when(care_non_smoker_logic, 'non/ex-smoker')
    .otherwise(None).alias('smoker_status')
)

# drop null smoker status
df_care_smoking_status = df_care_smoking_status_with_missing.where(F.col('smoker_status').isNotNull())

df_smoker_care = df_care_smoking_status.select(
    'UniqPregID', 'Person_ID_mother_DEID', 'smoker_status', 'smoker_status_date'
)

In [0]:
finding_smoker_logic = F.col('FindingCode').isin(SmokingFindingCode)
finding_non_smoker_logic = F.col('FindingCode').isin(NonSmokingFindingCode)
finding_cols = ['UniqPregID', 'Person_ID_mother_DEID', F.col('FindingDate').alias('smoker_status_date')]

df_smoker_finding = df_fo.select(
    *finding_cols,
    F.when(finding_smoker_logic, 'smoker')
    .when(finding_non_smoker_logic, 'non/ex-smoker')
    .otherwise(None).alias('smoker_status')
).where(F.col('smoker_status').isNotNull())

obs_smoker_logic = (
    ((F.col('obscode').isin(cigsperday_obscode)) & (F.col('obsvalue').try_cast('float') > 0)) |
    ((F.col('obscode').isin(carbon_monoxide_obscodes)) & (F.upper(F.col('ucumunit')).like('%ppm%')) & (F.col('obsvalue').try_cast('float') >= 4))
)
obs_non_smoker_logic = (
    (F.col('obscode') == NonSmokingObsCode) |
    ((F.col('obscode').isin(cigsperday_obscode)) & (F.col('obsvalue').try_cast('float') == 0)) |
    ((F.col('obscode').isin(carbon_monoxide_obscodes)) & (F.upper(F.col('ucumunit')).like('%ppm%')) & (F.col('obsvalue').try_cast('float') < 4))
)

obs_cols = ['UniqPregID', 'Person_ID_mother_DEID', F.col('obsdate').alias('smoker_status_date')]

df_smoker_obs = df_fo.select(
    *finding_cols,
    F.when(obs_smoker_logic, 'smoker')
    .when(obs_non_smoker_logic, 'non/ex-smoker')
    .otherwise(None).alias('smoker_status')
).where(F.col('smoker_status').isNotNull())

In [0]:
df_smoking_statuses = (
    df_smoker_care
    .unionByName(df_smoker_finding)
    .unionByName(df_smoker_obs)
).drop_duplicates()

In [0]:
df_smoking = (
    df_master_final
    .join(
        df_smoking_statuses.drop(*["Person_ID_mother_DEID"]),
        on = 'UniqPregID',
        how = 'left'
    )
    .withColumn(
        "is_smoker",
        when(F.col("smoker_status") == "smoker", 1).when(col('smoker_status') == 'non/ex-smoker', 0).otherwise(None)
    ).withColumn(
        "is_current_window",
        F.when(
            (F.col("smoker_status") == "smoker") &
            (F.col("smoker_status_date").between(
                F.col("last_period_date"),
                F.col("ld_hosp_disch_date")
            )),
            1
        ).otherwise(0)
    )
    .groupBy("UniqPregID").agg(
        F.max("is_smoker").alias("ever_smoker_raw"),
        F.max("is_current_window").alias("current_smoker_raw"),
        F.count("smoker_status").alias("has_records")
    )
    .withColumn(
        "ever_smoker",
        F.when(F.col("has_records") == 0, None)
        .otherwise(F.col("ever_smoker_raw"))
    ).withColumn(
        "current_smoker",
        F.when(F.col("has_records") == 0, None)
        .otherwise(F.col("current_smoker_raw"))
    )
)

In [0]:
df_smoking = (
    df_smoking
    .withColumns({
        "ever_smoker": when(col("ever_smoker").isNull(), 0).otherwise(col("ever_smoker")),
        "current_smoker": when(col("current_smoker").isNull(), 0).otherwise(col("current_smoker"))
    })
)

In [0]:
df_master_final = df_master_final.join(
    df_smoking,
    on='UniqPregID',
    how="left"
)

# Update Contacts Attended

In [0]:
# contacts attended vs non-attended
df_dbp = spark.table(f'dsa_712819_x8g2j.dsa_712819_x8g2j_collab.msds_v2_demographics_booking_and_pregnancy_all_years')

df_contacts = (
    df_dbp
    .select('UniqPregID', 'Person_id_mother_deid', 'carecontact_attended_count','carecontact_notattended_count')
    .withColumn(
        "total_scheduled_contacts", 
        when((col("carecontact_attended_count").isNotNull() & col("carecontact_notattended_count").isNotNull()), col("carecontact_attended_count") + col("carecontact_notattended_count"))
        .when((col("carecontact_attended_count").isNotNull() & col("carecontact_notattended_count").isNull()), col("carecontact_attended_count"))
        .when((col("carecontact_attended_count").isNull() & col("carecontact_notattended_count").isNotNull()), col("carecontact_notattended_count"))
        .otherwise(None)
    )
    .withColumn("contacts_attended_pct", when((col("carecontact_attended_count").isNotNull() & col("total_scheduled_contacts").isNotNull() & (col("total_scheduled_contacts") != 0)), 100 * col("carecontact_attended_count") / col("total_scheduled_contacts")).otherwise(0)) # Otherwise used to be None
    .groupBy("UniqPregID").agg(
        F.max("total_scheduled_contacts").alias("total_scheduled_contacts"),
        F.max("contacts_attended_pct").alias("contacts_attended_pct")
    )
)

In [0]:
df_master_final = (
    df_master_final
    .join(
        df_contacts, on="UniqPregID", how="left"
    )
)

# IMD Ranking

In [0]:
df_demographic = spark.sql(
    "SELECT DISTINCT * FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.msds_v2_demographics_booking_and_pregnancy_all_years"
).select("UniqPregID", "LSOAMother2011")

df_lsoas = df_demographic.groupBy('UniqPregID').agg(
    first('LSOAMother2011', ignorenulls=True).alias('LSOAMother2011')
)

df_master_lsoa = (
    df_master_final
    .join(
        df_lsoas,
        on="UniqPregID",
        how="left"
    )
)

In [0]:
df_imd = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.english_imd_2019")

df_master_imd = (
    df_master_lsoa
    .join(
        df_imd,
        df_master_lsoa.LSOAMother2011 == df_imd.LSOA_code_2011,
        how = "left"
    )
    #.filter(col("IMD_Rank").isNotNull())
    .drop(*["LSOA_code_2011"])
)


In [0]:
IMD_rank = [
    row["IMD_Rank"]
    for row in df_master_imd.filter(col("IMD_Rank").isNotNull()).select("IMD_Rank").distinct().collect()
]
IMD_rank = [int(r) for r in IMD_rank]
total_neighbourhoods = np.max(IMD_rank) # 32844

df_master_final = (
    df_master_imd
    .withColumn(
        "IMD_Rank_500",
        floor((col("IMD_Rank").cast("int") - 1) / 500)
    )
    .withColumn(
        "IMD_Rank_scaled",
        ((100 - (100 * col("IMD_Rank").cast("int") / F.lit(total_neighbourhoods)))/10)
    )
    .withColumn(
        "IMD_Rank_10",
        when((col("IMD_Rank_scaled") == 0), 1)
        .otherwise(ceil(col("IMD_Rank_scaled")))
    )
)

# Stratification

In [0]:
df_demographics = spark.sql(
    "SELECT DISTINCT * FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.msds_v2_demographics_booking_and_pregnancy_all_years"
)

In [0]:
# Group 1: 
df_demographics = (
    df_demographics
    .withColumns({
        "previouscaesareansections": col("previouscaesareansections").try_cast('int'),
        "previouslivebirths": col("previouslivebirths").try_cast('int'),
        "previouslosseslessthan24weeks": col("previouslosseslessthan24weeks").try_cast('int'),
        "previousstillbirths": col("previousstillbirths").try_cast('int')
    })
    .groupBy("UniqPregID")
    .agg(*[
        max('previouscaesareansections').alias('num_prev_csections'),
        max('previouslivebirths').alias('num_prev_births'),
        max('previouslosseslessthan24weeks').alias('num_prev_24w_losses'),
        max('previousstillbirths').alias('num_prev_still')
    ])
    .withColumns({
        "previous_live_birth": when(col('num_prev_births') >= 1, 1).otherwise(0),
        "previous_still_birth":when(col('num_prev_still') >= 1, 1).otherwise(0),
    })
    .withColumns({
        "group_1": when(((col("previous_live_birth") == 0) & (col("previous_still_birth") == 0)), 1).otherwise(0),
        "group_2": when(((col("previous_live_birth") != 0) | (col("previous_still_birth") != 0)), 1).otherwise(0),
    })
    .select("UniqPregID", "group_1", "group_2")
)

df_master_final = (
    df_master_final
    .join(
        df_demographics,
        on="UniqPregID"
    )
    .withColumn("group", when(col("group_1") == 1, 1).when(col("group_2") == 1, 2).otherwise(None))
)

# Save dataset

In [0]:
select_cols = [
    "UniqPregID", "Person_ID_DEID", "group", "birth_year", "ld_hosp_org_site_id",

    # m1
    'deprived_reg', 'IMD_Rank_scaled',

    # m2
    'booking_age_u35', 'booking_age_35_40', 'booking_age_o40', 'ethnic_white_reg', 'ethnic_black_reg', 'ethnic_south_asian_reg', 'ethnic_mixed_reg', 'ethnic_other_reg', 'lang_not_english',

    # m3
    'mother_bmi_u185', 'mother_bmi_185_249', 'mother_bmi_250_299', 'mother_bmi_o300',
    "ever_substance_use", "ever_smoking", "ever_smoker", "current_smoker", 
    'folic_acid_while_pregnant',     
    'contacts_attended', "total_scheduled_contacts", "contacts_attended_pct",

    'Prior_Preeclampsia', 'Prior_Gestational_Diabetes',
    'Prior_Infectious_Disease', 'Prior_Neoplasm', 'Prior_Blood_or_Immune_Disease', 'Prior_Endocrine_or_Metabolic_Disease', 'Prior_Mental_Disorder', 'Prior_Nervous_System_Disease', 'Prior_Circulatory_Disease', 'Prior_Respiratory_Disease', 'Prior_Gastrointestinal_Disease', 'Prior_Musculoskeletal_Disease', 'Prior_Genitourinary_Disease', 'Prior_Congenital_Abnormality',

    # outcomes
    "Preeclampsia_onset", "Preeclampsia_during_this_pregnancy", "Preeclampsia_early_onset", "Preeclampsia_late_onset"
]

df_master_select = df_master_final.select(select_cols)

In [0]:
df_master_select = (
    df_master_select    # For Cox PH model
    .withColumn(
        "Preeclampsia_onset", 
        when((col("Preeclampsia_during_this_pregnancy") != 1), F.lit((42*7 + 1))).otherwise(col("Preeclampsia_onset"))
    )
)

In [0]:
df_master_select.toPandas().to_csv("/Workspace/Shared/regression_datasets/full_master_df_for_regression.xlsx")

In [0]:
write_table_name = "msds_diag_busy_filtered_final_imputed_reduced_timed_ind_ranked_stage"

df_master_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{write_table_name}")